# Week 8 — A2C, GAE, TRPO · Workbook

Eight graded exercises. Reference implementations live in `project.solutions`.


**Teaching companion:** [Worked examples, derivations, analogies, and self-checks](teaching_guide.md).


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, torch, matplotlib.pyplot as plt
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
from project import solutions as sol
from project.train import train


## Exercise 1 (★) — Prove identity (†)

Verify numerically that for an $n$-step advantage we have
$\hat A^{(n)}_t = \sum_{k=0}^{n-1} \gamma^k \delta_{t+k}$ where
$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$.

> **Hint**: compute both sides on a 6-step toy with random rewards and value-net outputs.


In [ ]:
rng = np.random.default_rng(0); T = 6; gamma = 0.9
r = rng.normal(size=T); v = rng.normal(size=T); last_v = float(rng.normal())
# Direct n-step (n=4):
n = 4
direct = sol.nstep_advantage(r, v, gamma, n, bootstrap=last_v)[0]
# Via sum of deltas:
ext_v = np.concatenate([v, [last_v]])
deltas = r + gamma * ext_v[1:] - ext_v[:-1]
via_delta = sum(gamma**k * deltas[k] for k in range(n))
print('direct[0] =', direct, ' via delta =', via_delta)
assert abs(direct - via_delta) < 1e-9


## Exercise 2 (★) — `compute_gae`

Implement GAE and verify on a 10-step random rollout that $\lambda=0$ recovers TD
and $\lambda=1$ recovers MC only for a completed episode; at an unfinished rollout boundary it retains the final bootstrap.

> **Hint**: iterate from the end with the recurrence `adv_t = delta_t + gamma*lam*(1-done)*adv_{t+1}`.


In [ ]:
T = 10; gamma=0.99; rng = np.random.default_rng(1)
r = rng.normal(size=T); v = rng.normal(size=T); last_v = 0.0
adv_td, _ = sol.compute_gae(r, v, np.zeros(T), last_v, gamma=gamma, lam=0.0)
adv_mc, _ = sol.compute_gae(r, v, np.zeros(T), last_v, gamma=gamma, lam=1.0)
# TD: matches r + gamma*v_{t+1} - v_t
td = r.copy(); td[:-1] += gamma * v[1:] - v[:-1]; td[-1] += gamma * last_v - v[-1]
assert np.allclose(adv_td, td, atol=1e-6)
print('TD ok')
# MC: return-to-go minus value
G = np.zeros(T); run = last_v
for t in range(T-1, -1, -1):
    run = r[t] + gamma*run; G[t] = run
assert np.allclose(adv_mc, G - v, atol=1e-5)
print('MC ok')


## Exercise 3 (★★) — Limiting cases

Show analytically (in your notebook in markdown) that GAE($\lambda{=}0$) collapses
to one-step TD advantage, and GAE($\lambda{=}1$) collapses to MC advantage for a completed episode, and includes the final bootstrap for an incomplete rollout.
Cite the geometric-series step.


In [ ]:
# Analysis. With lambda=0, the GAE sum sum_l (gamma*lam)^l delta_{t+l} keeps only l=0:
# A_t = delta_t. With lambda=1, the sum is the Monte Carlo cumulative TD residual,
# which by telescoping equals G_t - V(s_t).
# (Both verified empirically in E2.)
print('analytic check noted')


## Exercise 4 (★★) — Train A2C+GAE on LunarLanderContinuous

Run a short training pass and confirm the learning curve trends upward.
We use a small budget (40 updates) so the notebook stays papermill-fast.

> **Hint**: `train(seed=0, n_updates=40)` returns episode returns; plot a moving average.


In [ ]:
out = train(seed=0, n_updates=40, num_envs=4, rollout_steps=64)
r = np.array(out.episode_returns) if out.episode_returns else np.zeros(1)
print('n_episodes:', len(r), 'final mean:', round(float(r[-10:].mean() if len(r)>=10 else r.mean()), 1))
if len(r) >= 5:
    fig, ax = plt.subplots(figsize=(5.5, 3.0))
    ax.plot(r, color=RL_PALETTE['teal']); ax.set_xlabel('episode'); ax.set_ylabel('return')
    fig.tight_layout(); plt.show()


## Exercise 5 (★★) — $\lambda$ sweep

Train with $\lambda \in \{0, 0.5, 0.95, 1.0\}$ for a small budget; report final returns.
(The full sweep is the figure in the notes.)


In [ ]:
results = {}
for lam in [0.0, 0.5, 0.95, 1.0]:
    out = train(seed=0, n_updates=20, lam=lam, num_envs=4, rollout_steps=64)
    r = np.array(out.episode_returns) if out.episode_returns else np.array([0.0])
    results[lam] = float(r[-10:].mean() if len(r)>=10 else r.mean())
print('final returns by lambda:', {k: round(v,1) for k,v in results.items()})


## Exercise 6 (★★) — Fisher matrix of a Gaussian policy

For $\pi_\theta(a|s) = \mathcal{N}(\mu_\theta(s), \sigma_\theta(s)^2)$ with parameters
$\theta = (\mu, \log\sigma)$, show that the Fisher is block-diagonal with
$F_{\mu\mu} = 1/\sigma^2$ and $F_{ll} = 2$ (here $l = \log\sigma$). Verify numerically via
Monte Carlo at $\mu=0, \sigma=1$.


In [ ]:
torch.manual_seed(0)
mu = torch.zeros(1, requires_grad=True)
log_sigma = torch.zeros(1, requires_grad=True)
N = 8000
F_mm = 0.0; F_ll = 0.0; F_ml = 0.0
for _ in range(N):
    eps = torch.randn(1)
    # Sample with parameters fixed (detach a) so log-prob is a function of params alone.
    a = (mu + log_sigma.exp() * eps).detach()
    logp = -0.5 * ((a - mu) / log_sigma.exp())**2 - log_sigma - 0.5*np.log(2*np.pi)
    gmu, gls = torch.autograd.grad(logp.sum(), [mu, log_sigma])
    F_mm += float(gmu * gmu); F_ll += float(gls * gls); F_ml += float(gmu * gls)
F_mm /= N; F_ll /= N; F_ml /= N
print('F_mu_mu =', round(F_mm, 3), '(expect 1.0)')
print('F_l_l   =', round(F_ll, 3), '(expect 2.0)')
print('F_mu_l  =', round(F_ml, 3), '(expect 0.0)')
assert abs(F_mm - 1.0) < 0.15 and abs(F_ll - 2.0) < 0.25 and abs(F_ml) < 0.2


## Exercise 7 (★★★) — Conjugate gradient for $F^{-1}g$

Build a 4x4 SPD `A` and a random `b`, then solve via `conjugate_gradient` and
verify against `np.linalg.solve`.


In [ ]:
rng = np.random.default_rng(2)
n = 4; M = rng.normal(size=(n, n)); A = M @ M.T + np.eye(n) * 0.5
b = rng.normal(size=n)
x_cg = sol.conjugate_gradient(lambda v: A @ v, b, n_iters=20)
x_ref = np.linalg.solve(A, b)
print('||x_cg - x_ref||_2 =', float(np.linalg.norm(x_cg - x_ref)))
assert np.allclose(x_cg, x_ref, atol=1e-6)


## Exercise 8 (★★★, *meaningful objective*) — Trust-region weekly retrain

Given logged matching data `(s, a, r)` and a current policy with parameters $\mu_\text{old}$,
distinguish pointwise and average KL constraints. Pinsker gives $\|p-q\|_1\le\sqrt{2\delta}$, while a single broker/event probability changes by at most $\sqrt{\delta/2}\approx2.24$ percentage points at $\delta=10^{-3}$. An average-KL constraint bounds average event changes under the **same fixed context distribution**, not every context separately.

Demonstrate by sampling $\mu_\text{new}$ a tiny perturbation of $\mu_\text{old}$, computing the
KL. For Gaussian policies, densities on a grid are not action probabilities: integrate probability over defined action regions, or use a categorical policy to demonstrate broker shares. The existing Gaussian cell illustrates KL calculation only.


In [ ]:
torch.manual_seed(0)
mu_old = torch.zeros(2); log_std = torch.zeros(2)
# small perturbation
mu_new = mu_old + 0.05
kl = sol.diagonal_gaussian_kl(mu_old[None], log_std[None], mu_new[None], log_std[None]).item()
print('KL =', round(kl, 6))
import math
bound = math.sqrt(kl / 2)
print('Pinsker TV bound =', round(bound, 4))
# Action-space probability shift, sampled on a grid
from torch.distributions import Normal
x = torch.linspace(-2, 2, 200).unsqueeze(-1)
p_old = Normal(mu_old, log_std.exp()).log_prob(x).exp().prod(-1)
p_new = Normal(mu_new, log_std.exp()).log_prob(x).exp().prod(-1)
max_shift = (p_new - p_old).abs().max().item()
print('max pointwise shift in joint density =', round(max_shift, 4))


### Recap

You derived and verified GAE, trained A2C on a continuous-control env, computed
the Fisher for a Gaussian policy, and solved a Fisher--vector system by conjugate
gradient. Next week: PPO replaces the second-order TRPO machinery with a clipped
first-order surrogate and avoids the conjugate-gradient and backtracking machinery; runtime gains depend on the implementation and task.


<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>Exercise 1 (★) — Prove identity (†)</summary>

> **Hint**: compute both sides on a 6-step toy with random rewards and value-net outputs.

</details>

<details>
<summary>Exercise 2 (★) — `compute_gae`</summary>

> **Hint**: iterate from the end with the recurrence `adv_t = delta_t + gamma*lam*(1-done)*adv_{t+1}`.

</details>

<details>
<summary>Exercise 3 (★★) — Limiting cases</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 4 (★★) — Train A2C+GAE on LunarLanderContinuous</summary>

> **Hint**: `train(seed=0, n_updates=40)` returns episode returns; plot a moving average.

</details>

<details>
<summary>Exercise 5 (★★) — $\lambda$ sweep</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 6 (★★) — Fisher matrix of a Gaussian policy</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 7 (★★★) — Conjugate gradient for $F^{-1}g$</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 8 (★★★, *meaningful objective*) — Trust-region weekly retrain</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

After solving, change one input or assumption and predict the result before
running it. If a cell calls `project.solutions`, passing it verifies the
reference demonstration; test your own implementation separately.
